# Sign Language Detection - Kaggle Training

Attach an ASL alphabet directory dataset, enable GPU, run all cells, then download `/kaggle/working/sign_language_models.zip`.

In [ ]:
import os, json, zipfile, random
from pathlib import Path
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED=42; EPOCHS=20; BATCH_SIZE=64; IMG_SIZE=(64,64)
MODEL_DIR=Path('/kaggle/working/models'); MODEL_DIR.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

def find_class_dir():
    best=None; best_count=0
    for root, dirs, files in os.walk('/kaggle/input'):
        class_dirs=[d for d in dirs if len(d)<=15]
        if len(class_dirs)>best_count:
            best=root; best_count=len(class_dirs)
    if not best or best_count<10: raise FileNotFoundError('Could not find ASL class directories')
    return best
DATA_DIR=find_class_dir(); print('Using:',DATA_DIR)
train_ds=keras.utils.image_dataset_from_directory(DATA_DIR,validation_split=0.2,subset='training',seed=SEED,image_size=IMG_SIZE,batch_size=BATCH_SIZE,label_mode='categorical')
val_ds=keras.utils.image_dataset_from_directory(DATA_DIR,validation_split=0.2,subset='validation',seed=SEED,image_size=IMG_SIZE,batch_size=BATCH_SIZE,label_mode='categorical')
labels=train_ds.class_names
norm=layers.Rescaling(1./255)
train_ds=train_ds.map(lambda x,y:(norm(x),y)).prefetch(tf.data.AUTOTUNE)
val_ds=val_ds.map(lambda x,y:(norm(x),y)).prefetch(tf.data.AUTOTUNE)
model=keras.Sequential([layers.Input(shape=(64,64,3)),layers.Conv2D(32,3,activation='relu'),layers.MaxPooling2D(),layers.Conv2D(64,3,activation='relu'),layers.MaxPooling2D(),layers.Conv2D(128,3,activation='relu'),layers.MaxPooling2D(),layers.Flatten(),layers.Dense(256,activation='relu'),layers.Dropout(0.5),layers.Dense(len(labels),activation='softmax')],name='sign_language_cnn')
model.compile(keras.optimizers.Adam(1e-3),loss='categorical_crossentropy',metrics=['accuracy'])
hist=model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS,callbacks=[keras.callbacks.EarlyStopping(patience=4,restore_best_weights=True)])
loss,acc=model.evaluate(val_ds,verbose=0); print('val_accuracy',acc)
model.save(MODEL_DIR/'sign_language_cnn.keras')
(MODEL_DIR/'sign_language_config.json').write_text(json.dumps({'seed':SEED,'class_labels':labels,'input_shape':[64,64,3],'validation_accuracy':float(acc)},indent=2))
zip_path='/kaggle/working/sign_language_models.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in MODEL_DIR.iterdir(): z.write(p,arcname=p.name)
print('Download:',zip_path); print(sorted(p.name for p in MODEL_DIR.iterdir()))